# 04 Modeling — Disaster Type Classification

This notebook develops and compares machine learning models for
multi-class disaster type classification using the processed historical
disaster dataset.

Five classification algorithms are evaluated under four modeling
strategies:

1. Class Weight
2. SMOTETomek
3. SMOTETomek + Hyperparameter Tuning (Pre-CV Resampling)
4. SMOTETomek + Hyperparameter Tuning (Pipeline)

The primary evaluation metric is Macro F1 because the dataset contains
imbalanced disaster classes.

In [77]:
import pandas as pd
import numpy as np

# Load processed dataset
data = pd.read_csv("../data/processed/disaster_processed_feature_selected.csv")

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (17474, 246)


,start_month,event_duration_log_norm,magnitude_missing,region_Africa,region_Americas,region_Asia,region_Europe,region_Oceania,magnitude_scale_Km2,magnitude_scale_Kph,...,country_Venezuela (Bolivarian Republic of),country_Viet Nam,country_Wallis and Futuna Islands,country_Yemen,country_Yemen Arab Republic,country_Yugoslavia,country_Zambia,country_Zimbabwe,magnitude_scaled,disaster_type_encoded
0,9.0,0.0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0.329509,6
1,1.0,0.0,1,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0.000000,4
2,1.0,0.0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.000000,2
3,7.0,0.0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0.117902,1
4,1.0,0.0,1,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0.000000,0


STEP 2 — Define X and y

In [78]:
# Separate features and target

X = data.drop(columns=["disaster_type_encoded"])
y = data["disaster_type_encoded"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (17474, 245)
Target shape: (17474,)


STEP 3 — Verify that unwanted features are gone

In [79]:
# Check unwanted features

total_death_features = [
    col for col in X.columns
    if "death" in col.lower()
]

subregion_features = [
    col for col in X.columns
    if "subregion" in col.lower()
]

print("Total Death related features:", total_death_features)
print("Subregion related features:", subregion_features)

Total Death related features: []
Subregion related features: []


STEP 4 — Check missing values again

In [80]:
print("Missing values in X:")
print(X.isnull().sum()[X.isnull().sum() > 0])

print("\nMissing values in y:")
print(y.isnull().sum())

Missing values in X:
Series([], dtype: int64)

Missing values in y:
0


STEP 5 — Train/Test Split

In [81]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (13979, 245)
Testing shape: (3495, 245)


STEP 6 — Verify class distribution

In [82]:
train_distribution = y_train.value_counts().sort_index()
test_distribution = y_test.value_counts().sort_index()

distribution_table = pd.DataFrame({
    "Training": train_distribution,
    "Testing": test_distribution
})

distribution_table["Training %"] = (
    distribution_table["Training"] / len(y_train) * 100
).round(2)

distribution_table["Testing %"] = (
    distribution_table["Testing"] / len(y_test) * 100
).round(2)

distribution_table

,Training,Testing,Training %,Testing %
disaster_type_encoded,,,,
0,643,160,4.60,4.58
1,1321,330,9.45,9.44
2,1298,324,9.29,9.27
3,584,146,4.18,4.18
4,4946,1237,35.38,35.39
5,734,184,5.25,5.26
6,4042,1011,28.91,28.93
7,411,103,2.94,2.95


STEP 7 — Define the evaluation function

In [83]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def calculate_metrics(y_true, y_pred):

    return {
        "Accuracy": accuracy_score(y_true, y_pred),

        "Precision Macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "Recall Macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "F1 Macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "Precision Weighted": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "Recall Weighted": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "F1 Weighted": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }

STEP 8 — Define the 5 models

In [11]:
%pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.6 MB 1.5 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [84]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [85]:
models = {

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    ),

    "SVC": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
}

Create the results list

In [86]:
# ============================================================
# EXPERIMENT 1: CLASS WEIGHT
# ============================================================

results = []

Create Class Weight versions of the models

In [87]:
class_weight_models = {

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    ),

    "SVC": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    )
}

In [88]:
# ============================================================
# EXPERIMENT 1 — CLASS WEIGHT
# ============================================================

class_weight_results = []

for model_name in [
    "Random Forest",
    "Logistic Regression",
    "SVC"
]:
    
    print("\n" + "=" * 70)
    print(f" {model_name} — Class Weight")
    print("=" * 70)

    # Get model
    model = class_weight_models[model_name]

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    metrics = calculate_metrics(y_test, y_pred)

    # Store results
    class_weight_results.append({
        "Model": model_name,
        "Method": "Class Weight",
        **metrics
    })

    # Display results
    print(f"Accuracy           : {metrics['Accuracy']:.4f}")
    print(f"Precision (Macro)  : {metrics['Precision Macro']:.4f}")
    print(f"Recall (Macro)     : {metrics['Recall Macro']:.4f}")
    print(f"F1 (Macro)         : {metrics['F1 Macro']:.4f}")
    print(f"Precision (Weighted): {metrics['Precision Weighted']:.4f}")
    print(f"Recall (Weighted)   : {metrics['Recall Weighted']:.4f}")
    print(f"F1 (Weighted)       : {metrics['F1 Weighted']:.4f}")

# Convert results to DataFrame
class_weight_results_df = pd.DataFrame(class_weight_results)

print("\n" + "=" * 70)
print("CLASS WEIGHT — SUMMARY")
print("=" * 70)

display(
    class_weight_results_df.round(4)
)


 Random Forest — Class Weight
Accuracy           : 0.9227
Precision (Macro)  : 0.8478
Recall (Macro)     : 0.8480
F1 (Macro)         : 0.8479
Precision (Weighted): 0.9227
Recall (Weighted)   : 0.9227
F1 (Weighted)       : 0.9227

 Logistic Regression — Class Weight
Accuracy           : 0.8235
Precision (Macro)  : 0.7733
Recall (Macro)     : 0.8467
F1 (Macro)         : 0.7794
Precision (Weighted): 0.9085
Recall (Weighted)   : 0.8235
F1 (Weighted)       : 0.8465

 SVC — Class Weight
Accuracy           : 0.8415
Precision (Macro)  : 0.7769
Recall (Macro)     : 0.8369
F1 (Macro)         : 0.7862
Precision (Weighted): 0.9040
Recall (Weighted)   : 0.8415
F1 (Weighted)       : 0.8611

CLASS WEIGHT — SUMMARY


,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,Class Weight,0.9227,0.8478,0.8480,0.8479,0.9227,0.9227,0.9227
1,Logistic Regression,Class Weight,0.8235,0.7733,0.8467,0.7794,0.9085,0.8235,0.8465
2,SVC,Class Weight,0.8415,0.7769,0.8369,0.7862,0.9040,0.8415,0.8611


In [19]:
# ============================================================
# LIGHTGBM — CREATE SAFE FEATURE NAMES
# ============================================================

import re

# Make a copy so the original X remains unchanged
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()

# Replace special characters with "_"
X_train_lgb.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', str(col))
    for col in X_train_lgb.columns
]

X_test_lgb.columns = X_train_lgb.columns

print("Original feature count:", X_train.shape[1])
print("LightGBM feature count:", X_train_lgb.shape[1])

# Check that feature names are now safe
special_features = [
    col for col in X_train_lgb.columns
    if not re.match(r'^[A-Za-z0-9_]+$', col)
]

print("Unsafe feature names:", special_features)

Original feature count: 245
LightGBM feature count: 245
Unsafe feature names: []


In [20]:
# ============================================================
# LIGHTGBM + CLASS WEIGHT
# ============================================================

lgb_model = LGBMClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(X_train_lgb, y_train)

y_pred_lgb = lgb_model.predict(X_test_lgb)

lgb_metrics = calculate_metrics(y_test, y_pred_lgb)

print("=" * 70)
print(" LightGBM — Class Weight")
print("=" * 70)

for metric, value in lgb_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

class_weight_results.append({
    "Model": "LightGBM",
    "Method": "Class Weight",
    **lgb_metrics
})

 LightGBM — Class Weight
Accuracy              : 0.9102
Precision Macro       : 0.8281
Recall Macro          : 0.8697
F1 Macro              : 0.8437
Precision Weighted    : 0.9268
Recall Weighted       : 0.9102
F1 Weighted           : 0.9163


In [21]:
class_weight_results_df = pd.DataFrame(class_weight_results)

display(
    class_weight_results_df.round(4)
)

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,Class Weight,0.9227,0.8478,0.8480,0.8479,0.9227,0.9227,0.9227
1,Logistic Regression,Class Weight,0.8232,0.7732,0.8466,0.7792,0.9084,0.8232,0.8462
2,SVC,Class Weight,0.8415,0.7769,0.8369,0.7862,0.9040,0.8415,0.8611
3,LightGBM,Class Weight,0.9102,0.8281,0.8697,0.8437,0.9268,0.9102,0.9163


Import SMOTETomek

In [22]:
# ============================================================
# EXPERIMENT 2 — SMOTETOMEK
# ============================================================

from imblearn.combine import SMOTETomek

Create the SMOTETomek object

In [23]:
smotetomek = SMOTETomek(
    random_state=42
)

X_train_st, y_train_st = smotetomek.fit_resample(
    X_train,
    y_train
)

print("Original training shape:", X_train.shape)
print("After SMOTETomek:", X_train_st.shape)

print("\nClass distribution after SMOTETomek:")
print(pd.Series(y_train_st).value_counts().sort_index())

/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Original training shape: (13979, 245)
After SMOTETomek: (39350, 245)

Class distribution after SMOTETomek:
disaster_type_encoded
0    4885
1    4946
2    4927
3    4946
4    4865
5    4927
6    4946
7    4908
Name: count, dtype: int64


In [24]:
smotetomek_models = {

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    ),

    "SVC": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
}

In [25]:
# ============================================================
# TRAIN & EVALUATE — SMOTETOMEK
# ============================================================

smotetomek_results = []

for model_name, model in smotetomek_models.items():

    print("\n" + "=" * 70)
    print(f" {model_name} — SMOTETomek")
    print("=" * 70)

    # Use safe feature names for LightGBM
    if model_name == "LightGBM":
        X_train_model = X_train_st.copy()
        X_test_model = X_test.copy()

        X_train_model.columns = [
            re.sub(r'[^A-Za-z0-9_]+', '_', str(col))
            for col in X_train_model.columns
        ]

        X_test_model.columns = X_train_model.columns

    else:
        X_train_model = X_train_st
        X_test_model = X_test

    # Train
    model.fit(X_train_model, y_train_st)

    # Predict on ORIGINAL test set
    y_pred = model.predict(X_test_model)

    # Calculate metrics
    metrics = calculate_metrics(y_test, y_pred)

    # Save results
    smotetomek_results.append({
        "Model": model_name,
        "Method": "SMOTETomek",
        **metrics
    })

    # Display
    print(f"Accuracy            : {metrics['Accuracy']:.4f}")
    print(f"Precision (Macro)   : {metrics['Precision Macro']:.4f}")
    print(f"Recall (Macro)      : {metrics['Recall Macro']:.4f}")
    print(f"F1 (Macro)          : {metrics['F1 Macro']:.4f}")
    print(f"Precision (Weighted): {metrics['Precision Weighted']:.4f}")
    print(f"Recall (Weighted)   : {metrics['Recall Weighted']:.4f}")
    print(f"F1 (Weighted)       : {metrics['F1 Weighted']:.4f}")


 Random Forest — SMOTETomek
Accuracy            : 0.9167
Precision (Macro)   : 0.8369
Recall (Macro)      : 0.8680
F1 (Macro)          : 0.8499
Precision (Weighted): 0.9261
Recall (Weighted)   : 0.9167
F1 (Weighted)       : 0.9204

 Logistic Regression — SMOTETomek


/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:336: 

Accuracy            : 0.8661
Precision (Macro)   : 0.7828
Recall (Macro)      : 0.8315
F1 (Macro)          : 0.7958
Precision (Weighted): 0.8998
Recall (Weighted)   : 0.8661
F1 (Weighted)       : 0.8781

 LightGBM — SMOTETomek
Accuracy            : 0.9193
Precision (Macro)   : 0.8401
Recall (Macro)      : 0.8536
F1 (Macro)          : 0.8463
Precision (Weighted): 0.9229
Recall (Weighted)   : 0.9193
F1 (Weighted)       : 0.9209

 XGBoost — SMOTETomek
Accuracy            : 0.9144
Precision (Macro)   : 0.8344
Recall (Macro)      : 0.8725
F1 (Macro)          : 0.8497
Precision (Weighted): 0.9260
Recall (Weighted)   : 0.9144
F1 (Weighted)       : 0.9187

 SVC — SMOTETomek
Accuracy            : 0.8741
Precision (Macro)   : 0.7941
Recall (Macro)      : 0.8511
F1 (Macro)          : 0.8095
Precision (Weighted): 0.9110
Recall (Weighted)   : 0.8741
F1 (Weighted)       : 0.8865


In [26]:
smotetomek_results_df = pd.DataFrame(smotetomek_results)

display(
    smotetomek_results_df.round(4)
)

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,SMOTETomek,0.9167,0.8369,0.8680,0.8499,0.9261,0.9167,0.9204
1,Logistic Regression,SMOTETomek,0.8661,0.7828,0.8315,0.7958,0.8998,0.8661,0.8781
2,LightGBM,SMOTETomek,0.9193,0.8401,0.8536,0.8463,0.9229,0.9193,0.9209
3,XGBoost,SMOTETomek,0.9144,0.8344,0.8725,0.8497,0.9260,0.9144,0.9187
4,SVC,SMOTETomek,0.8741,0.7941,0.8511,0.8095,0.9110,0.8741,0.8865


Experiment - 3

Resample the training data

In [27]:
# ============================================================
# EXPERIMENT 3
# SMOTETomek + Hyperparameter Tuning
# PRE-CV RESAMPLING
# ============================================================

from imblearn.combine import SMOTETomek

smotetomek_tuned = SMOTETomek(
    random_state=42
)

X_train_tuned, y_train_tuned = smotetomek_tuned.fit_resample(
    X_train,
    y_train
)

print("Original training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nAfter SMOTETomek:")
print("X_train_tuned:", X_train_tuned.shape)
print("y_train_tuned:", y_train_tuned.shape)

print("\nClass distribution after SMOTETomek:")
print(
    pd.Series(y_train_tuned)
    .value_counts()
    .sort_index()
)

/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Original training data:
X_train: (13979, 245)
y_train: (13979,)

After SMOTETomek:
X_train_tuned: (39350, 245)
y_train_tuned: (39350,)

Class distribution after SMOTETomek:
disaster_type_encoded
0    4885
1    4946
2    4927
3    4946
4    4865
5    4927
6    4946
7    4908
Name: count, dtype: int64


Prepare safe feature names for LightGBM

In [28]:
# ============================================================
# SAFE FEATURE NAMES FOR LIGHTGBM
# ============================================================

import re

X_train_tuned_lgb = X_train_tuned.copy()
X_test_lgb = X_test.copy()

X_train_tuned_lgb.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', str(col))
    for col in X_train_tuned_lgb.columns
]

X_test_lgb.columns = X_train_tuned_lgb.columns

print("Number of features:", X_train_tuned_lgb.shape[1])

unsafe_features = [
    col for col in X_train_tuned_lgb.columns
    if not re.match(r'^[A-Za-z0-9_]+$', col)
]

print("Unsafe LightGBM features:", unsafe_features)

Number of features: 245
Unsafe LightGBM features: []


Import RandomizedSearchCV

In [29]:
# ============================================================
# HYPERPARAMETER TUNING
# ============================================================

from sklearn.model_selection import RandomizedSearchCV

Define Random Forest tuning

In [30]:
# ============================================================
# RANDOM FOREST — PARAMETER SEARCH
# ============================================================

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False]
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_grid,
    n_iter=20,
    scoring="f1_macro",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [31]:
# ============================================================
# RUN RANDOM FOREST TUNING
# ============================================================

print("Starting Random Forest RandomizedSearchCV...")

rf_search.fit(
    X_train_tuned,
    y_train_tuned
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(rf_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{rf_search.best_score_:.4f}")

Starting Random Forest RandomizedSearchCV...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Tuning completed!

Best Parameters:
{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'bootstrap': True}

Best Cross-Validation F1-Macro:
0.9493


Evaluate the tuned Random Forest

In [32]:
# ============================================================
# FINAL TEST EVALUATION — TUNED RANDOM FOREST
# ============================================================

best_rf = rf_search.best_estimator_

y_pred_rf = best_rf.predict(X_test)

rf_tuned_metrics = calculate_metrics(
    y_test,
    y_pred_rf
)

print("=" * 70)
print(" RANDOM FOREST + SMOTETomek + TUNING")
print(" Pre-CV Resampling")
print("=" * 70)

for metric, value in rf_tuned_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 RANDOM FOREST + SMOTETomek + TUNING
 Pre-CV Resampling
Accuracy              : 0.9159
Precision Macro       : 0.8366
Recall Macro          : 0.8644
F1 Macro              : 0.8484
Precision Weighted    : 0.9239
Recall Weighted       : 0.9159
F1 Weighted           : 0.9191


Classification report

In [33]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_rf,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.5421    0.6438    0.5886       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9040    0.9012    0.9026       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9385    0.8884    0.9128      1237
          Landslide     0.8226    0.8315    0.8270       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.4855    0.6505    0.5560       103

           accuracy                         0.9159      3495
          macro avg     0.8366    0.8644    0.8484      3495
       weighted avg     0.9239    0.9159    0.9191      3495



In [34]:
# ============================================================
# SAVE EXPERIMENT 3 RESULT
# ============================================================

experiment3_results = []

experiment3_results.append({
    "Model": "Random Forest",
    "Method": "SMOTETomek + Tuning (Pre-CV)",
    **rf_tuned_metrics
})

experiment3_results_df = pd.DataFrame(
    experiment3_results
)

display(
    experiment3_results_df.round(4)
)

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,SMOTETomek + Tuning (Pre-CV),0.9159,0.8366,0.8644,0.8484,0.9239,0.9159,0.9191


In [35]:
# ============================================================
# EXPERIMENT 3 — LOGISTIC REGRESSION
# SMOTETomek + Tuning (Pre-CV Resampling)
# ============================================================

lr = LogisticRegression(
    max_iter=3000,
    random_state=42
)

lr_param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "solver": ["lbfgs", "liblinear"],
    "class_weight": [None]
}

lr_search = RandomizedSearchCV(
    estimator=lr,
    param_distributions=lr_param_grid,
    n_iter=10,
    scoring="f1_macro",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [36]:
print("Starting Logistic Regression tuning...")

lr_search.fit(
    X_train_tuned,
    y_train_tuned
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(lr_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{lr_search.best_score_:.4f}")

Starting Logistic Regression tuning...
Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:336: 


Tuning completed!

Best Parameters:
{'solver': 'lbfgs', 'class_weight': None, 'C': 10}

Best Cross-Validation F1-Macro:
0.8941


In [37]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

best_lr = lr_search.best_estimator_

y_pred_lr = best_lr.predict(X_test)

lr_tuned_metrics = calculate_metrics(
    y_test,
    y_pred_lr
)

print("=" * 70)
print(" LOGISTIC REGRESSION + SMOTETomek + TUNING")
print(" Pre-CV Resampling")
print("=" * 70)

for metric, value in lr_tuned_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 LOGISTIC REGRESSION + SMOTETomek + TUNING
 Pre-CV Resampling
Accuracy              : 0.8718
Precision Macro       : 0.7842
Recall Macro          : 0.8238
F1 Macro              : 0.7955
Precision Weighted    : 0.8986
Recall Weighted       : 0.8718
F1 Weighted           : 0.8819


/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [38]:
print(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.3604    0.4437    0.3978       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9167    0.8827    0.8994       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9067    0.7939    0.8466      1237
          Landslide     0.8020    0.8587    0.8294       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.2877    0.6117    0.3913       103

           accuracy                         0.8718      3495
          macro avg     0.7842    0.8238    0.7955      3495
       weighted avg     0.8986    0.8718    0.8819      3495



In [39]:
# ============================================================
# EXPERIMENT 3 — LIGHTGBM
# ============================================================

lgb = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [-1, 10, 20, 30],
    "num_leaves": [15, 31, 50, 75],
    "min_child_samples": [10, 20, 30, 50],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

lgb_search = RandomizedSearchCV(
    estimator=lgb,
    param_distributions=lgb_param_grid,
    n_iter=20,
    scoring="f1_macro",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [40]:
print("Starting LightGBM tuning...")

lgb_search.fit(
    X_train_tuned_lgb,
    y_train_tuned
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(lgb_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{lgb_search.best_score_:.4f}")

Starting LightGBM tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Tuning completed!

Best Parameters:
{'subsample': 0.8, 'num_leaves': 75, 'n_estimators': 200, 'min_child_samples': 10, 'max_depth': 20, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

Best Cross-Validation F1-Macro:
0.9511


In [41]:
best_lgb = lgb_search.best_estimator_

y_pred_lgb = best_lgb.predict(X_test_lgb)

lgb_tuned_metrics = calculate_metrics(
    y_test,
    y_pred_lgb
)

print("=" * 70)
print(" LIGHTGBM + SMOTETomek + TUNING")
print(" Pre-CV Resampling")
print("=" * 70)

for metric, value in lgb_tuned_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 LIGHTGBM + SMOTETomek + TUNING
 Pre-CV Resampling
Accuracy              : 0.9225
Precision Macro       : 0.8461
Recall Macro          : 0.8565
F1 Macro              : 0.8510
Precision Weighted    : 0.9250
Recall Weighted       : 0.9225
F1 Weighted           : 0.9236


In [42]:
print(
    classification_report(
        y_test,
        y_pred_lgb,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.5731    0.6125    0.5921       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9185    0.9043    0.9114       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9301    0.9143    0.9221      1237
          Landslide     0.8298    0.8478    0.8387       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.5175    0.5728    0.5438       103

           accuracy                         0.9225      3495
          macro avg     0.8461    0.8565    0.8510      3495
       weighted avg     0.9250    0.9225    0.9236      3495



In [43]:
# ============================================================
# EXPERIMENT 3 — XGBOOST
# ============================================================

xgb = XGBClassifier(
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss"
)

xgb_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7, 10],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=xgb_param_grid,
    n_iter=20,
    scoring="f1_macro",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [44]:
print("Starting XGBoost tuning...")

xgb_search.fit(
    X_train_tuned,
    y_train_tuned
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(xgb_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{xgb_search.best_score_:.4f}")

Starting XGBoost tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/Users/chaw/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Tuning completed!

Best Parameters:
{'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.2, 'colsample_bytree': 0.8}

Best Cross-Validation F1-Macro:
0.9386


In [45]:
best_xgb = xgb_search.best_estimator_

y_pred_xgb = best_xgb.predict(X_test)

xgb_tuned_metrics = calculate_metrics(
    y_test,
    y_pred_xgb
)

print("=" * 70)
print(" XGBOOST + SMOTETomek + TUNING")
print(" Pre-CV Resampling")
print("=" * 70)

for metric, value in xgb_tuned_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 XGBOOST + SMOTETomek + TUNING
 Pre-CV Resampling
Accuracy              : 0.9167
Precision Macro       : 0.8382
Recall Macro          : 0.8655
F1 Macro              : 0.8500
Precision Weighted    : 0.9246
Recall Weighted       : 0.9167
F1 Weighted           : 0.9198


In [46]:
print(
    classification_report(
        y_test,
        y_pred_xgb,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.5468    0.6937    0.6116       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9117    0.8920    0.9017       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9385    0.8884    0.9128      1237
          Landslide     0.8125    0.8478    0.8298       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.4960    0.6019    0.5439       103

           accuracy                         0.9167      3495
          macro avg     0.8382    0.8655    0.8500      3495
       weighted avg     0.9246    0.9167    0.9198      3495



In [50]:
# ============================================================
# EXPERIMENT 3 — SVC
# SMOTETomek + Tuning (Pre-CV Resampling)
# ============================================================

svc = SVC(
    kernel="rbf",
    probability=False,
    random_state=42
)

svc_param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1],
    "class_weight": [None]
}

svc_search = RandomizedSearchCV(
    estimator=svc,
    param_distributions=svc_param_grid,
    n_iter=8,
    scoring="f1_macro",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [51]:
print("Starting SVC tuning...")

svc_search.fit(
    X_train_tuned,
    y_train_tuned
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(svc_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{svc_search.best_score_:.4f}")

Starting SVC tuning...
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Tuning completed!

Best Parameters:
{'gamma': 'scale', 'class_weight': None, 'C': 100}

Best Cross-Validation F1-Macro:
0.9232


In [52]:
best_svc = svc_search.best_estimator_

y_pred_svc = best_svc.predict(X_test)

svc_tuned_metrics = calculate_metrics(
    y_test,
    y_pred_svc
)

print("=" * 70)
print(" SVC + SMOTETomek + TUNING")
print(" Pre-CV Resampling")
print("=" * 70)

for metric, value in svc_tuned_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 SVC + SMOTETomek + TUNING
 Pre-CV Resampling
Accuracy              : 0.9062
Precision Macro       : 0.8226
Recall Macro          : 0.8476
F1 Macro              : 0.8303
Precision Weighted    : 0.9154
Recall Weighted       : 0.9062
F1 Weighted           : 0.9093


In [53]:
print(
    classification_report(
        y_test,
        y_pred_svc,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.5385    0.4813    0.5083       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9336    0.8673    0.8992       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9191    0.8812    0.8997      1237
          Landslide     0.7923    0.8913    0.8389       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.3977    0.6602    0.4964       103

           accuracy                         0.9062      3495
          macro avg     0.8226    0.8476    0.8303      3495
       weighted avg     0.9154    0.9062    0.9093      3495



In [54]:
# ============================================================
# EXPERIMENT 3 — SUMMARY
# ============================================================

experiment3_results = [
    {
        "Model": "Random Forest",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **rf_tuned_metrics
    },
    {
        "Model": "Logistic Regression",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **lr_tuned_metrics
    },
    {
        "Model": "LightGBM",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **lgb_tuned_metrics
    },
    {
        "Model": "XGBoost",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **xgb_tuned_metrics
    },
    {
        "Model": "SVC",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **svc_tuned_metrics
    }
]

experiment3_results_df = pd.DataFrame(
    experiment3_results
)

display(
    experiment3_results_df.round(4)
)

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,SMOTETomek + Tuning (Pre-CV),0.9159,0.8366,0.8644,0.8484,0.9239,0.9159,0.9191
1,Logistic Regression,SMOTETomek + Tuning (Pre-CV),0.8718,0.7842,0.8238,0.7955,0.8986,0.8718,0.8819
2,LightGBM,SMOTETomek + Tuning (Pre-CV),0.9225,0.8461,0.8565,0.8510,0.9250,0.9225,0.9236
3,XGBoost,SMOTETomek + Tuning (Pre-CV),0.9167,0.8382,0.8655,0.8500,0.9246,0.9167,0.9198
4,SVC,SMOTETomek + Tuning (Pre-CV),0.9062,0.8226,0.8476,0.8303,0.9154,0.9062,0.9093


In [55]:
# ============================================================
# EXPERIMENT 3 — FINAL SUMMARY
# SMOTETomek + Tuning (Pre-CV Resampling)
# ============================================================

experiment3_results = [
    {
        "Model": "Random Forest",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **rf_tuned_metrics
    },
    {
        "Model": "Logistic Regression",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **lr_tuned_metrics
    },
    {
        "Model": "LightGBM",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **lgb_tuned_metrics
    },
    {
        "Model": "XGBoost",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **xgb_tuned_metrics
    },
    {
        "Model": "SVC",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        **svc_tuned_metrics
    }
]

experiment3_results_df = pd.DataFrame(experiment3_results)

# Sort by main metric: F1-Macro
experiment3_results_df = experiment3_results_df.sort_values(
    by="F1 Macro",
    ascending=False
).reset_index(drop=True)

display(
    experiment3_results_df.round(4)
)

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,LightGBM,SMOTETomek + Tuning (Pre-CV),0.9225,0.8461,0.8565,0.8510,0.9250,0.9225,0.9236
1,XGBoost,SMOTETomek + Tuning (Pre-CV),0.9167,0.8382,0.8655,0.8500,0.9246,0.9167,0.9198
2,Random Forest,SMOTETomek + Tuning (Pre-CV),0.9159,0.8366,0.8644,0.8484,0.9239,0.9159,0.9191
3,SVC,SMOTETomek + Tuning (Pre-CV),0.9062,0.8226,0.8476,0.8303,0.9154,0.9062,0.9093
4,Logistic Regression,SMOTETomek + Tuning (Pre-CV),0.8718,0.7842,0.8238,0.7955,0.8986,0.8718,0.8819


Import Pipeline and SMOTETomek

In [56]:
# ============================================================
# EXPERIMENT 4
# SMOTETomek + Tuning (Pipeline)
# ============================================================

from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek
from sklearn.model_selection import RandomizedSearchCV

print("Experiment 4 setup ready.")

Experiment 4 setup ready.


Create the Pipeline components

In [57]:
# ============================================================
# SMOTETomek
# ============================================================

smotetomek = SMOTETomek(
    random_state=42
)

print("SMOTETomek created.")

SMOTETomek created.


In [58]:
# ============================================================
# RANDOM FOREST PIPELINE
# ============================================================

rf_pipeline = Pipeline([
    ("smotetomek", SMOTETomek(random_state=42)),
    ("random_forest", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline_param_grid = {
    "random_forest__n_estimators": [100, 200, 300, 500],
    "random_forest__max_depth": [None, 10, 20, 30, 40],
    "random_forest__min_samples_split": [2, 5, 10],
    "random_forest__min_samples_leaf": [1, 2, 4],
    "random_forest__max_features": ["sqrt", "log2"],
    "random_forest__bootstrap": [True, False]
}

In [59]:
# ============================================================
# RANDOM FOREST — RANDOMIZED SEARCH
# ============================================================

rf_pipeline_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_pipeline_param_grid,
    n_iter=20,
    scoring="f1_macro",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Random Forest Pipeline search created.")

Random Forest Pipeline search created.


In [60]:
# ============================================================
# RUN RANDOM FOREST PIPELINE TUNING
# ============================================================

print("Starting Random Forest Pipeline tuning...")

rf_pipeline_search.fit(
    X_train,
    y_train
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(rf_pipeline_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{rf_pipeline_search.best_score_:.4f}")

Starting Random Forest Pipeline tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/s


Tuning completed!

Best Parameters:
{'random_forest__n_estimators': 100, 'random_forest__min_samples_split': 5, 'random_forest__min_samples_leaf': 1, 'random_forest__max_features': 'log2', 'random_forest__max_depth': 40, 'random_forest__bootstrap': True}

Best Cross-Validation F1-Macro:
0.8521


In [62]:
# ============================================================
# RANDOM FOREST — FINAL TEST EVALUATION
# ============================================================

best_rf_pipeline = rf_pipeline_search.best_estimator_

y_pred_rf_pipeline = best_rf_pipeline.predict(X_test)

rf_pipeline_metrics = calculate_metrics(
    y_test,
    y_pred_rf_pipeline
)

print("=" * 70)
print(" RANDOM FOREST + SMOTETomek + TUNING (PIPELINE)")
print("=" * 70)

for metric, value in rf_pipeline_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 RANDOM FOREST + SMOTETomek + TUNING (PIPELINE)
Accuracy              : 0.9156
Precision Macro       : 0.8372
Recall Macro          : 0.8694
F1 Macro              : 0.8507
Precision Weighted    : 0.9251
Recall Weighted       : 0.9156
F1 Weighted           : 0.9193


In [63]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        y_test,
        y_pred_rf_pipeline,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.5419    0.6875    0.6061       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9122    0.8981    0.9051       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9405    0.8812    0.9098      1237
          Landslide     0.8211    0.8478    0.8342       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.4818    0.6408    0.5500       103

           accuracy                         0.9156      3495
          macro avg     0.8372    0.8694    0.8507      3495
       weighted avg     0.9251    0.9156    0.9193      3495



In [64]:
# ============================================================
# LOGISTIC REGRESSION PIPELINE
# ============================================================

lr_pipeline = Pipeline([
    ("smotetomek", SMOTETomek(random_state=42)),
    ("logistic_regression", LogisticRegression(
        max_iter=3000,
        random_state=42
    ))
])

lr_pipeline_param_grid = {
    "logistic_regression__C": [
        0.001, 0.01, 0.1, 1, 10, 100
    ],
    "logistic_regression__solver": [
        "lbfgs", "liblinear"
    ]
}

lr_pipeline_search = RandomizedSearchCV(
    estimator=lr_pipeline,
    param_distributions=lr_pipeline_param_grid,
    n_iter=10,
    scoring="f1_macro",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [65]:
print("Starting Logistic Regression Pipeline tuning...")

lr_pipeline_search.fit(
    X_train,
    y_train
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(lr_pipeline_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{lr_pipeline_search.best_score_:.4f}")

Starting Logistic Regression Pipeline tuning...
Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/s


Tuning completed!

Best Parameters:
{'logistic_regression__solver': 'liblinear', 'logistic_regression__C': 10}

Best Cross-Validation F1-Macro:
0.8107


In [66]:
# ============================================================
# LOGISTIC REGRESSION — FINAL TEST EVALUATION
# ============================================================

best_lr_pipeline = lr_pipeline_search.best_estimator_

y_pred_lr_pipeline = best_lr_pipeline.predict(X_test)

lr_pipeline_metrics = calculate_metrics(
    y_test,
    y_pred_lr_pipeline
)

print("=" * 70)
print(" LOGISTIC REGRESSION + SMOTETomek + TUNING (PIPELINE)")
print("=" * 70)

for metric, value in lr_pipeline_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 LOGISTIC REGRESSION + SMOTETomek + TUNING (PIPELINE)
Accuracy              : 0.8692
Precision Macro       : 0.7837
Recall Macro          : 0.8261
F1 Macro              : 0.7956
Precision Weighted    : 0.8979
Recall Weighted       : 0.8692
F1 Weighted           : 0.8798


/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [67]:
print(
    classification_report(
        y_test,
        y_pred_lr_pipeline,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.3663    0.4625    0.4088       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9191    0.8765    0.8973       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9049    0.7842    0.8402      1237
          Landslide     0.7950    0.8641    0.8281       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.2844    0.6214    0.3902       103

           accuracy                         0.8692      3495
          macro avg     0.7837    0.8261    0.7956      3495
       weighted avg     0.8979    0.8692    0.8798      3495



In [68]:
# ============================================================
# SAFE FEATURE NAMES FOR LIGHTGBM
# ============================================================

import re

X_train_lgb_pipeline = X_train.copy()
X_test_lgb_pipeline = X_test.copy()

X_train_lgb_pipeline.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', str(col))
    for col in X_train_lgb_pipeline.columns
]

X_test_lgb_pipeline.columns = X_train_lgb_pipeline.columns

print("LightGBM feature names cleaned.")
print("Number of features:", X_train_lgb_pipeline.shape[1])

LightGBM feature names cleaned.
Number of features: 245


In [69]:
# ============================================================
# LIGHTGBM PIPELINE
# ============================================================

lgb_pipeline = Pipeline([
    ("smotetomek", SMOTETomek(random_state=42)),
    ("lightgbm", LGBMClassifier(
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

lgb_pipeline_param_grid = {
    "lightgbm__n_estimators": [100, 200, 300, 500],
    "lightgbm__learning_rate": [0.01, 0.05, 0.1],
    "lightgbm__max_depth": [-1, 10, 20, 30],
    "lightgbm__num_leaves": [15, 31, 50, 75],
    "lightgbm__min_child_samples": [10, 20, 30, 50],
    "lightgbm__subsample": [0.8, 1.0],
    "lightgbm__colsample_bytree": [0.8, 1.0]
}

lgb_pipeline_search = RandomizedSearchCV(
    estimator=lgb_pipeline,
    param_distributions=lgb_pipeline_param_grid,
    n_iter=20,
    scoring="f1_macro",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [70]:
print("Starting LightGBM Pipeline tuning...")

lgb_pipeline_search.fit(
    X_train_lgb_pipeline,
    y_train
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(lgb_pipeline_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{lgb_pipeline_search.best_score_:.4f}")

Starting LightGBM Pipeline tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/s


Tuning completed!

Best Parameters:
{'lightgbm__subsample': 1.0, 'lightgbm__num_leaves': 50, 'lightgbm__n_estimators': 100, 'lightgbm__min_child_samples': 20, 'lightgbm__max_depth': 20, 'lightgbm__learning_rate': 0.1, 'lightgbm__colsample_bytree': 1.0}

Best Cross-Validation F1-Macro:
0.8580


In [71]:
# ============================================================
# LIGHTGBM — FINAL TEST EVALUATION
# ============================================================

best_lgb_pipeline = lgb_pipeline_search.best_estimator_

y_pred_lgb_pipeline = best_lgb_pipeline.predict(
    X_test_lgb_pipeline
)

lgb_pipeline_metrics = calculate_metrics(
    y_test,
    y_pred_lgb_pipeline
)

print("=" * 70)
print(" LIGHTGBM + SMOTETomek + TUNING (PIPELINE)")
print("=" * 70)

for metric, value in lgb_pipeline_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 LIGHTGBM + SMOTETomek + TUNING (PIPELINE)
Accuracy              : 0.9190
Precision Macro       : 0.8397
Recall Macro          : 0.8632
F1 Macro              : 0.8500
Precision Weighted    : 0.9255
Recall Weighted       : 0.9190
F1 Weighted           : 0.9217


In [72]:
print(
    classification_report(
        y_test,
        y_pred_lgb_pipeline,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.5668    0.6625    0.6110       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9148    0.8951    0.9048       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9383    0.8981    0.9178      1237
          Landslide     0.8168    0.8478    0.8320       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.4806    0.6019    0.5345       103

           accuracy                         0.9190      3495
          macro avg     0.8397    0.8632    0.8500      3495
       weighted avg     0.9255    0.9190    0.9217      3495



In [73]:
# ============================================================
# XGBOOST PIPELINE
# ============================================================

xgb_pipeline = Pipeline([
    ("smotetomek", SMOTETomek(random_state=42)),
    ("xgboost", XGBClassifier(
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    ))
])

xgb_pipeline_param_grid = {
    "xgboost__n_estimators": [100, 200, 300, 500],
    "xgboost__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "xgboost__max_depth": [3, 5, 7, 10],
    "xgboost__min_child_weight": [1, 3, 5],
    "xgboost__subsample": [0.8, 1.0],
    "xgboost__colsample_bytree": [0.8, 1.0]
}

xgb_pipeline_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_pipeline_param_grid,
    n_iter=20,
    scoring="f1_macro",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [74]:
print("Starting XGBoost Pipeline tuning...")

xgb_pipeline_search.fit(
    X_train,
    y_train
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(xgb_pipeline_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{xgb_pipeline_search.best_score_:.4f}")

Starting XGBoost Pipeline tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/s


Tuning completed!

Best Parameters:
{'xgboost__subsample': 0.8, 'xgboost__n_estimators': 300, 'xgboost__min_child_weight': 3, 'xgboost__max_depth': 5, 'xgboost__learning_rate': 0.2, 'xgboost__colsample_bytree': 0.8}

Best Cross-Validation F1-Macro:
0.8511


In [75]:
# ============================================================
# XGBOOST — FINAL TEST EVALUATION
# ============================================================

best_xgb_pipeline = xgb_pipeline_search.best_estimator_

y_pred_xgb_pipeline = best_xgb_pipeline.predict(X_test)

xgb_pipeline_metrics = calculate_metrics(
    y_test,
    y_pred_xgb_pipeline
)

print("=" * 70)
print(" XGBOOST + SMOTETomek + TUNING (PIPELINE)")
print("=" * 70)

for metric, value in xgb_pipeline_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 XGBOOST + SMOTETomek + TUNING (PIPELINE)
Accuracy              : 0.9133
Precision Macro       : 0.8321
Recall Macro          : 0.8662
F1 Macro              : 0.8461
Precision Weighted    : 0.9238
Recall Weighted       : 0.9133
F1 Weighted           : 0.9174


In [76]:
print(
    classification_report(
        y_test,
        y_pred_xgb_pipeline,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.5354    0.6625    0.5922       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9196    0.8827    0.9008       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9404    0.8795    0.9089      1237
          Landslide     0.8030    0.8641    0.8325       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.4583    0.6408    0.5344       103

           accuracy                         0.9133      3495
          macro avg     0.8321    0.8662    0.8461      3495
       weighted avg     0.9238    0.9133    0.9174      3495



In [77]:
# ============================================================
# SVC PIPELINE
# ============================================================

svc_pipeline = Pipeline([
    ("smotetomek", SMOTETomek(random_state=42)),
    ("svc", SVC(
        kernel="rbf",
        probability=False,
        random_state=42
    ))
])

svc_pipeline_param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": [
        "scale",
        "auto",
        0.001,
        0.01,
        0.1
    ]
}

svc_pipeline_search = RandomizedSearchCV(
    estimator=svc_pipeline,
    param_distributions=svc_pipeline_param_grid,
    n_iter=8,
    scoring="f1_macro",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [78]:
print("Starting SVC Pipeline tuning...")

svc_pipeline_search.fit(
    X_train,
    y_train
)

print("\nTuning completed!")

print("\nBest Parameters:")
print(svc_pipeline_search.best_params_)

print("\nBest Cross-Validation F1-Macro:")
print(f"{svc_pipeline_search.best_score_:.4f}")

Starting SVC Pipeline tuning...
Fitting 3 folds for each of 8 candidates, totalling 24 fits


/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/chaw/Library/Python/3.9/lib/python/s


Tuning completed!

Best Parameters:
{'svc__gamma': 'scale', 'svc__C': 100}

Best Cross-Validation F1-Macro:
0.8385


In [79]:
# ============================================================
# SVC — FINAL TEST EVALUATION
# ============================================================

best_svc_pipeline = svc_pipeline_search.best_estimator_

y_pred_svc_pipeline = best_svc_pipeline.predict(X_test)

svc_pipeline_metrics = calculate_metrics(
    y_test,
    y_pred_svc_pipeline
)

print("=" * 70)
print(" SVC + SMOTETomek + TUNING (PIPELINE)")
print("=" * 70)

for metric, value in svc_pipeline_metrics.items():
    print(f"{metric:<22}: {value:.4f}")

 SVC + SMOTETomek + TUNING (PIPELINE)
Accuracy              : 0.9062
Precision Macro       : 0.8226
Recall Macro          : 0.8476
F1 Macro              : 0.8303
Precision Weighted    : 0.9154
Recall Weighted       : 0.9062
F1 Weighted           : 0.9093


In [80]:
print(
    classification_report(
        y_test,
        y_pred_svc_pipeline,
        target_names=[
            "Drought",
            "Earthquake",
            "Epidemic",
            "Extreme Temperature",
            "Flood",
            "Landslide",
            "Storm",
            "Wildfire"
        ],
        digits=4
    )
)

                     precision    recall  f1-score   support

            Drought     0.5385    0.4813    0.5083       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9336    0.8673    0.8992       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9191    0.8812    0.8997      1237
          Landslide     0.7923    0.8913    0.8389       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.3977    0.6602    0.4964       103

           accuracy                         0.9062      3495
          macro avg     0.8226    0.8476    0.8303      3495
       weighted avg     0.9154    0.9062    0.9093      3495



In [81]:
# ============================================================
# EXPERIMENT 4 — FINAL SUMMARY
# ============================================================

experiment4_results = [
    {
        "Model": "Random Forest",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        **rf_pipeline_metrics
    },
    {
        "Model": "Logistic Regression",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        **lr_pipeline_metrics
    },
    {
        "Model": "LightGBM",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        **lgb_pipeline_metrics
    },
    {
        "Model": "XGBoost",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        **xgb_pipeline_metrics
    },
    {
        "Model": "SVC",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        **svc_pipeline_metrics
    }
]

experiment4_results_df = pd.DataFrame(
    experiment4_results
)

experiment4_results_df = experiment4_results_df.sort_values(
    by="F1 Macro",
    ascending=False
).reset_index(drop=True)

display(
    experiment4_results_df.round(4)
)

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,SMOTETomek + Tuning (Pipeline),0.9156,0.8372,0.8694,0.8507,0.9251,0.9156,0.9193
1,LightGBM,SMOTETomek + Tuning (Pipeline),0.9190,0.8397,0.8632,0.8500,0.9255,0.9190,0.9217
2,XGBoost,SMOTETomek + Tuning (Pipeline),0.9133,0.8321,0.8662,0.8461,0.9238,0.9133,0.9174
3,SVC,SMOTETomek + Tuning (Pipeline),0.9062,0.8226,0.8476,0.8303,0.9154,0.9062,0.9093
4,Logistic Regression,SMOTETomek + Tuning (Pipeline),0.8692,0.7837,0.8261,0.7956,0.8979,0.8692,0.8798


In [83]:
# ============================================================
# EXPERIMENT 1 — RECREATE RESULTS
# Class Weight
# ============================================================

experiment1_results_df = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Method": "Class Weight",
        "Accuracy": 0.9227,
        "Precision Macro": 0.8478,
        "Recall Macro": 0.8480,
        "F1 Macro": 0.8479,
        "Precision Weighted": 0.9227,
        "Recall Weighted": 0.9227,
        "F1 Weighted": 0.9227
    },
    {
        "Model": "Logistic Regression",
        "Method": "Class Weight",
        "Accuracy": 0.8232,
        "Precision Macro": 0.7732,
        "Recall Macro": 0.8466,
        "F1 Macro": 0.7792,
        "Precision Weighted": 0.9084,
        "Recall Weighted": 0.8232,
        "F1 Weighted": 0.8462
    },
    {
        "Model": "SVC",
        "Method": "Class Weight",
        "Accuracy": 0.8415,
        "Precision Macro": 0.7769,
        "Recall Macro": 0.8369,
        "F1 Macro": 0.7862,
        "Precision Weighted": 0.9040,
        "Recall Weighted": 0.8415,
        "F1 Weighted": 0.8611
    },
    {
        "Model": "LightGBM",
        "Method": "Class Weight",
        "Accuracy": 0.9102,
        "Precision Macro": 0.8281,
        "Recall Macro": 0.8697,
        "F1 Macro": 0.8437,
        "Precision Weighted": 0.9268,
        "Recall Weighted": 0.9102,
        "F1 Weighted": 0.9163
    }
])

display(experiment1_results_df.round(4))

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,Class Weight,0.9227,0.8478,0.8480,0.8479,0.9227,0.9227,0.9227
1,Logistic Regression,Class Weight,0.8232,0.7732,0.8466,0.7792,0.9084,0.8232,0.8462
2,SVC,Class Weight,0.8415,0.7769,0.8369,0.7862,0.9040,0.8415,0.8611
3,LightGBM,Class Weight,0.9102,0.8281,0.8697,0.8437,0.9268,0.9102,0.9163


In [85]:
# ============================================================
# EXPERIMENT 2 — RECREATE RESULTS
# SMOTETomek
# ============================================================

experiment2_results_df = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Method": "SMOTETomek",
        "Accuracy": 0.9167,
        "Precision Macro": 0.8369,
        "Recall Macro": 0.8680,
        "F1 Macro": 0.8499,
        "Precision Weighted": 0.9261,
        "Recall Weighted": 0.9167,
        "F1 Weighted": 0.9204
    },
    {
        "Model": "Logistic Regression",
        "Method": "SMOTETomek",
        "Accuracy": 0.8661,
        "Precision Macro": 0.7828,
        "Recall Macro": 0.8315,
        "F1 Macro": 0.7958,
        "Precision Weighted": 0.8998,
        "Recall Weighted": 0.8661,
        "F1 Weighted": 0.8781
    },
    {
        "Model": "LightGBM",
        "Method": "SMOTETomek",
        "Accuracy": 0.9193,
        "Precision Macro": 0.8401,
        "Recall Macro": 0.8536,
        "F1 Macro": 0.8463,
        "Precision Weighted": 0.9229,
        "Recall Weighted": 0.9193,
        "F1 Weighted": 0.9209
    },
    {
        "Model": "XGBoost",
        "Method": "SMOTETomek",
        "Accuracy": 0.9144,
        "Precision Macro": 0.8344,
        "Recall Macro": 0.8725,
        "F1 Macro": 0.8497,
        "Precision Weighted": 0.9260,
        "Recall Weighted": 0.9144,
        "F1 Weighted": 0.9187
    },
    {
        "Model": "SVC",
        "Method": "SMOTETomek",
        "Accuracy": 0.8741,
        "Precision Macro": 0.7941,
        "Recall Macro": 0.8511,
        "F1 Macro": 0.8095,
        "Precision Weighted": 0.9110,
        "Recall Weighted": 0.8741,
        "F1 Weighted": 0.8865
    }
])

display(experiment2_results_df.round(4))

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,SMOTETomek,0.9167,0.8369,0.8680,0.8499,0.9261,0.9167,0.9204
1,Logistic Regression,SMOTETomek,0.8661,0.7828,0.8315,0.7958,0.8998,0.8661,0.8781
2,LightGBM,SMOTETomek,0.9193,0.8401,0.8536,0.8463,0.9229,0.9193,0.9209
3,XGBoost,SMOTETomek,0.9144,0.8344,0.8725,0.8497,0.9260,0.9144,0.9187
4,SVC,SMOTETomek,0.8741,0.7941,0.8511,0.8095,0.9110,0.8741,0.8865


In [86]:
# ============================================================
# EXPERIMENT 3 — RECREATE RESULTS
# SMOTETomek + Tuning (Pre-CV)
# ============================================================

experiment3_results_df = pd.DataFrame([
    {
        "Model": "LightGBM",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        "Accuracy": 0.9225,
        "Precision Macro": 0.8461,
        "Recall Macro": 0.8565,
        "F1 Macro": 0.8510,
        "Precision Weighted": 0.9250,
        "Recall Weighted": 0.9225,
        "F1 Weighted": 0.9236
    },
    {
        "Model": "XGBoost",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        "Accuracy": 0.9167,
        "Precision Macro": 0.8382,
        "Recall Macro": 0.8655,
        "F1 Macro": 0.8500,
        "Precision Weighted": 0.9246,
        "Recall Weighted": 0.9167,
        "F1 Weighted": 0.9198
    },
    {
        "Model": "Random Forest",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        "Accuracy": 0.9159,
        "Precision Macro": 0.8366,
        "Recall Macro": 0.8644,
        "F1 Macro": 0.8484,
        "Precision Weighted": 0.9239,
        "Recall Weighted": 0.9159,
        "F1 Weighted": 0.9191
    },
    {
        "Model": "SVC",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        "Accuracy": 0.9062,
        "Precision Macro": 0.8226,
        "Recall Macro": 0.8476,
        "F1 Macro": 0.8303,
        "Precision Weighted": 0.9154,
        "Recall Weighted": 0.9062,
        "F1 Weighted": 0.9093
    },
    {
        "Model": "Logistic Regression",
        "Method": "SMOTETomek + Tuning (Pre-CV)",
        "Accuracy": 0.8718,
        "Precision Macro": 0.7842,
        "Recall Macro": 0.8238,
        "F1 Macro": 0.7955,
        "Precision Weighted": 0.8986,
        "Recall Weighted": 0.8718,
        "F1 Weighted": 0.8819
    }
])

display(experiment3_results_df.round(4))

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,LightGBM,SMOTETomek + Tuning (Pre-CV),0.9225,0.8461,0.8565,0.8510,0.9250,0.9225,0.9236
1,XGBoost,SMOTETomek + Tuning (Pre-CV),0.9167,0.8382,0.8655,0.8500,0.9246,0.9167,0.9198
2,Random Forest,SMOTETomek + Tuning (Pre-CV),0.9159,0.8366,0.8644,0.8484,0.9239,0.9159,0.9191
3,SVC,SMOTETomek + Tuning (Pre-CV),0.9062,0.8226,0.8476,0.8303,0.9154,0.9062,0.9093
4,Logistic Regression,SMOTETomek + Tuning (Pre-CV),0.8718,0.7842,0.8238,0.7955,0.8986,0.8718,0.8819


In [87]:
# ============================================================
# EXPERIMENT 4 — RECREATE RESULTS
# SMOTETomek + Tuning (Pipeline)
# ============================================================

experiment4_results_df = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        "Accuracy": 0.9156,
        "Precision Macro": 0.8372,
        "Recall Macro": 0.8694,
        "F1 Macro": 0.8507,
        "Precision Weighted": 0.9251,
        "Recall Weighted": 0.9156,
        "F1 Weighted": 0.9193
    },
    {
        "Model": "LightGBM",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        "Accuracy": 0.9190,
        "Precision Macro": 0.8397,
        "Recall Macro": 0.8632,
        "F1 Macro": 0.8500,
        "Precision Weighted": 0.9255,
        "Recall Weighted": 0.9190,
        "F1 Weighted": 0.9217
    },
    {
        "Model": "XGBoost",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        "Accuracy": 0.9133,
        "Precision Macro": 0.8321,
        "Recall Macro": 0.8662,
        "F1 Macro": 0.8461,
        "Precision Weighted": 0.9238,
        "Recall Weighted": 0.9133,
        "F1 Weighted": 0.9174
    },
    {
        "Model": "SVC",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        "Accuracy": 0.9062,
        "Precision Macro": 0.8226,
        "Recall Macro": 0.8476,
        "F1 Macro": 0.8303,
        "Precision Weighted": 0.9154,
        "Recall Weighted": 0.9062,
        "F1 Weighted": 0.9093
    },
    {
        "Model": "Logistic Regression",
        "Method": "SMOTETomek + Tuning (Pipeline)",
        "Accuracy": 0.8692,
        "Precision Macro": 0.7837,
        "Recall Macro": 0.8261,
        "F1 Macro": 0.7956,
        "Precision Weighted": 0.8979,
        "Recall Weighted": 0.8692,
        "F1 Weighted": 0.8798
    }
])

display(experiment4_results_df.round(4))

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,SMOTETomek + Tuning (Pipeline),0.9156,0.8372,0.8694,0.8507,0.9251,0.9156,0.9193
1,LightGBM,SMOTETomek + Tuning (Pipeline),0.9190,0.8397,0.8632,0.8500,0.9255,0.9190,0.9217
2,XGBoost,SMOTETomek + Tuning (Pipeline),0.9133,0.8321,0.8662,0.8461,0.9238,0.9133,0.9174
3,SVC,SMOTETomek + Tuning (Pipeline),0.9062,0.8226,0.8476,0.8303,0.9154,0.9062,0.9093
4,Logistic Regression,SMOTETomek + Tuning (Pipeline),0.8692,0.7837,0.8261,0.7956,0.8979,0.8692,0.8798


In [88]:
# ============================================================
# MASTER COMPARISON — ALL 4 EXPERIMENTS
# ============================================================

all_results_df = pd.concat(
    [
        experiment1_results_df,
        experiment2_results_df,
        experiment3_results_df,
        experiment4_results_df
    ],
    ignore_index=True
)

# Sort by F1-Macro
all_results_df = all_results_df.sort_values(
    by="F1 Macro",
    ascending=False
).reset_index(drop=True)

display(
    all_results_df.round(4)
)

,Model,Method,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,LightGBM,SMOTETomek + Tuning (Pre-CV),0.9225,0.8461,0.8565,0.8510,0.9250,0.9225,0.9236
1,Random Forest,SMOTETomek + Tuning (Pipeline),0.9156,0.8372,0.8694,0.8507,0.9251,0.9156,0.9193
2,LightGBM,SMOTETomek + Tuning (Pipeline),0.9190,0.8397,0.8632,0.8500,0.9255,0.9190,0.9217
3,XGBoost,SMOTETomek + Tuning (Pre-CV),0.9167,0.8382,0.8655,0.8500,0.9246,0.9167,0.9198
4,Random Forest,SMOTETomek,0.9167,0.8369,0.8680,0.8499,0.9261,0.9167,0.9204
5,XGBoost,SMOTETomek,0.9144,0.8344,0.8725,0.8497,0.9260,0.9144,0.9187
6,Random Forest,SMOTETomek + Tuning (Pre-CV),0.9159,0.8366,0.8644,0.8484,0.9239,0.9159,0.9191
7,Random Forest,Class Weight,0.9227,0.8478,0.8480,0.8479,0.9227,0.9227,0.9227
8,LightGBM,SMOTETomek,0.9193,0.8401,0.8536,0.8463,0.9229,0.9193,0.9209
9,XGBoost,SMOTETomek + Tuning (Pipeline),0.9133,0.8321,0.8662,0.8461,0.9238,0.9133,0.9174


In [89]:
# ============================================================
# BASELINE — 5 MODELS
# No Class Weight / No SMOTETomek / No Tuning
# ============================================================

baseline_models = {

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    ),

    "SVC": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
}

In [91]:
# ============================================================
# LIGHTGBM — CLEAN FEATURE NAMES
# ============================================================

import re

# Copy the original data
X_train_lgb_baseline = X_train.copy()
X_test_lgb_baseline = X_test.copy()

# Clean feature names
def clean_feature_names(columns):
    cleaned = []

    for col in columns:
        col = str(col)

        # Replace anything except letters, numbers, and underscore
        col = re.sub(r'[^A-Za-z0-9_]', '_', col)

        # Remove repeated underscores
        col = re.sub(r'_+', '_', col)

        # Remove leading/trailing underscores
        col = col.strip('_')

        cleaned.append(col)

    return cleaned


cleaned_columns = clean_feature_names(
    X_train_lgb_baseline.columns
)

X_train_lgb_baseline.columns = cleaned_columns
X_test_lgb_baseline.columns = cleaned_columns

print("Original number of features:", X_train.shape[1])
print("Cleaned number of features:", X_train_lgb_baseline.shape[1])

print("\nExample cleaned feature names:")
print(X_train_lgb_baseline.columns[:20].tolist())

Original number of features: 245
Cleaned number of features: 245

Example cleaned feature names:
['start_month', 'event_duration_log_norm', 'magnitude_missing', 'region_Africa', 'region_Americas', 'region_Asia', 'region_Europe', 'region_Oceania', 'magnitude_scale_Km2', 'magnitude_scale_Kph', 'magnitude_scale_Moment_Magnitude', 'magnitude_scale_Unknown', 'magnitude_scale_Vaccinated', 'magnitude_scale_C', 'country_Afghanistan', 'country_Albania', 'country_Algeria', 'country_American_Samoa', 'country_Angola', 'country_Anguilla']


In [92]:
# ============================================================
# CHECK FEATURE NAMES
# ============================================================

print(
    "Train/Test feature names identical:",
    list(X_train_lgb_baseline.columns)
    == list(X_test_lgb_baseline.columns)
)

print(
    "Any special characters remaining:",
    any(
        re.search(r'[^A-Za-z0-9_]', str(col))
        for col in X_train_lgb_baseline.columns
    )
)

Train/Test feature names identical: True
Any special characters remaining: False


In [93]:
# ============================================================
# BASELINE — TRAIN AND PREDICT
# ============================================================

baseline_predictions = {}

for model_name, model in baseline_models.items():

    print("=" * 70)
    print(f"Training Baseline: {model_name}")
    print("=" * 70)

    # LightGBM needs cleaned feature names
    if model_name == "LightGBM":
        X_train_model = X_train_lgb_baseline
        X_test_model = X_test_lgb_baseline
    else:
        X_train_model = X_train
        X_test_model = X_test

    # Train
    model.fit(X_train_model, y_train)

    # Predict
    y_pred = model.predict(X_test_model)

    baseline_predictions[model_name] = y_pred

    print("Completed.")

Training Baseline: Random Forest
Completed.
Training Baseline: Logistic Regression
Completed.
Training Baseline: LightGBM
Completed.
Training Baseline: XGBoost
Completed.
Training Baseline: SVC
Completed.


In [101]:
# ============================================================
# BASELINE — CLASSIFICATION REPORTS
# ============================================================

class_names = [
    "Drought",
    "Earthquake",
    "Epidemic",
    "Extreme Temperature",
    "Flood",
    "Landslide",
    "Storm",
    "Wildfire"
]

for model_name, y_pred in baseline_predictions.items():

    print("\n")
    print("=" * 80)
    print(f"BASELINE — {model_name}")
    print("=" * 80)

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=class_names,
            digits=4
        )
    )



BASELINE — Random Forest
                     precision    recall  f1-score   support

            Drought     0.7603    0.5750    0.6548       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic     0.9161    0.9105    0.9133       324
Extreme Temperature     1.0000    1.0000    1.0000       146
              Flood     0.9149    0.9644    0.9390      1237
          Landslide     0.8387    0.8478    0.8432       184
              Storm     1.0000    1.0000    1.0000      1011
           Wildfire     0.6400    0.4660    0.5393       103

           accuracy                         0.9359      3495
          macro avg     0.8838    0.8455    0.8612      3495
       weighted avg     0.9320    0.9359    0.9327      3495



BASELINE — Logistic Regression
                     precision    recall  f1-score   support

            Drought     0.7945    0.3625    0.4979       160
         Earthquake     1.0000    1.0000    1.0000       330
           Epidemic  

/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/chaw/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [97]:
# ============================================================
# BASELINE — COMPARISON TABLE
# ============================================================

baseline_results = []

for model_name, y_pred in baseline_predictions.items():

    report = classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        output_dict=True
    )

    baseline_results.append({
        "Model": model_name,

        "Accuracy": accuracy_score(y_test, y_pred),

        "Precision Macro": report["macro avg"]["precision"],
        "Recall Macro": report["macro avg"]["recall"],
        "F1 Macro": report["macro avg"]["f1-score"],

        "Precision Weighted": report["weighted avg"]["precision"],
        "Recall Weighted": report["weighted avg"]["recall"],
        "F1 Weighted": report["weighted avg"]["f1-score"]
    })

baseline_results_df = pd.DataFrame(baseline_results)

display(
    baseline_results_df.round(4).sort_values(
        by="F1 Macro",
        ascending=False
    )
)

/Users/nurulhudaadamishaq/Documents/disaster-type-prediction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nurulhudaadamishaq/Documents/disaster-type-prediction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nurulhudaadamishaq/Documents/disaster-type-prediction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Us

,Model,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Random Forest,0.9359,0.8838,0.8455,0.8612,0.9320,0.9359,0.9327
2,LightGBM,0.9371,0.8834,0.8439,0.8599,0.9335,0.9371,0.9338
3,XGBoost,0.9336,0.8798,0.8306,0.8491,0.9288,0.9336,0.9291
1,Logistic Regression,0.9219,0.8850,0.7751,0.7929,0.9159,0.9219,0.9075
4,SVC,0.9142,0.8152,0.7404,0.7431,0.8947,0.9142,0.8896


In [103]:
# ============================================================
# DIFFICULT-CLASS COMPARISON
# Drought / Flood / Wildfire
# ============================================================

from sklearn.metrics import classification_report
import pandas as pd

class_names = [
    "Drought",
    "Earthquake",
    "Epidemic",
    "Extreme Temperature",
    "Flood",
    "Landslide",
    "Storm",
    "Wildfire"
]

models_to_compare = {
    "LightGBM - Pre-CV": (best_lgb, X_test),
    "XGBoost - Pre-CV": (best_xgb, X_test),
    "Random Forest - Pre-CV": (best_rf, X_test),
    "Random Forest - Pipeline": (best_rf_pipeline, X_test),
    "LightGBM - Pipeline": (best_lgb_pipeline, X_test_lgb_pipeline)
}

difficult_class_results = []

for model_name, (model, X_eval) in models_to_compare.items():

    y_pred = model.predict(X_eval)

    report = classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        output_dict=True
    )

    difficult_class_results.append({
        "Model": model_name,

        "Drought Precision": report["Drought"]["precision"],
        "Drought Recall": report["Drought"]["recall"],
        "Drought F1": report["Drought"]["f1-score"],

        "Flood Precision": report["Flood"]["precision"],
        "Flood Recall": report["Flood"]["recall"],
        "Flood F1": report["Flood"]["f1-score"],

        "Wildfire Precision": report["Wildfire"]["precision"],
        "Wildfire Recall": report["Wildfire"]["recall"],
        "Wildfire F1": report["Wildfire"]["f1-score"]
    })

difficult_class_df = pd.DataFrame(difficult_class_results)

display(difficult_class_df.round(4))

,Model,Drought Precision,Drought Recall,Drought F1,Flood Precision,Flood Recall,Flood F1,Wildfire Precision,Wildfire Recall,Wildfire F1
0,LightGBM - Pre-CV,0.5731,0.6125,0.5921,0.9301,0.9143,0.9221,0.5175,0.5728,0.5438
1,XGBoost - Pre-CV,0.5468,0.6938,0.6116,0.9385,0.8884,0.9128,0.4960,0.6019,0.5439
2,Random Forest - Pre-CV,0.5421,0.6438,0.5886,0.9385,0.8884,0.9128,0.4855,0.6505,0.5560
3,Random Forest - Pipeline,0.5419,0.6875,0.6061,0.9405,0.8812,0.9098,0.4818,0.6408,0.5500
4,LightGBM - Pipeline,0.5668,0.6625,0.6110,0.9383,0.8981,0.9178,0.4806,0.6019,0.5345
